04 - LLM report labeller
#
# The reports exist only at training time, so the labeller has no inference
# cost and can be as expensive as we like. That asymmetry is the main one in
# this competition, and the rule-based labeller does not exploit it: it tops
# out at 0.738 gold AUC because hand-written regex cannot cover 12 languages
# of hedged clinical prose.
#
# The 58 gold studies are scored first, through the same harness the
# rule-based labeller is scored by, so the two are directly comparable before
# any GPU time goes into the remaining 4349.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH = 8

In [ ]:
# Validation pass: 58 reports, a couple of minutes.
!cd {CODE} && python scripts/llm_label.py --gold-only --model {MODEL} \
    --batch {BATCH} --out /kaggle/working/llm_labels_gold.csv

In [ ]:
# Score the LLM labels against gold, side by side with the rule-based ones.
!cd {CODE} && python scripts/eval_labeler.py --llm /kaggle/working/llm_labels_gold.csv

In [ ]:
# Full corpus. Only worth the GPU time if the gold score above beat 0.738.
!cd {CODE} && python scripts/llm_label.py --model {MODEL} --batch {BATCH} \
    --out /kaggle/working/llm_labels.csv